# E3DTI training on Colab

Colab instead of Kaggle for one reason: **results go to Google Drive**, so a
dying session cannot take them with it. Three Kaggle sessions wiped
`/kaggle/working` in one afternoon and took finished runs with them.

Before running: **Runtime → Change runtime type → T4 GPU**.

Then **Runtime → Run all**. Cell 2 will ask you to authorise Drive access;
that is the only click needed. After that it is unattended — every run is
skipped if already finished and resumed if interrupted, so re-running this
notebook after a disconnect picks up exactly where it stopped.

In [ ]:
# 1. Confirm a GPU is attached before spending time on setup.
import torch

print('torch', torch.__version__)
print('cuda  ', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.')
print('device', torch.cuda.get_device_name(0))

In [ ]:
# 2. Mount Drive. This is the whole point: RUNS lives on Drive, so results
#    survive a disconnect, a timeout, or the runtime being recycled.
from google.colab import drive
drive.mount('/content/drive')

RUNS = '/content/drive/MyDrive/e3dti-runs'
import os
os.makedirs(RUNS, exist_ok=True)
print('results ->', RUNS)

In [ ]:
# 3. Dependencies. torch is already here; rdkit is not.
!pip install -q rdkit huggingface-hub

In [ ]:
# 4. Code. Cloned to local scratch, not Drive -- Drive is slow for many
#    small reads and the repo is disposable.
REPO = '/content/final-project-9'
!rm -rf {REPO}
!git clone -q --branch yogesh/drug-egnn https://github.com/XingqiaoLin/final-project-9.git {REPO}
!cd {REPO} && git log --oneline -1

In [ ]:
# 5. Correctness gates. docs/experiment_protocol.md requires these to pass
#    before any run counts.
!cd {REPO} && python -m unittest discover -s tests

In [ ]:
# 6. The matrix, unattended. One run at a time with --resume: a finished
#    run is skipped in milliseconds, an interrupted one restarts at its
#    next epoch. --budget-hours stops launching new runs before Colab's
#    own timer, so a session ends on a finished run.
#
#    --dry-run first to see the plan and what is already done.
!cd {REPO} && python scripts/run_kaggle_matrix.py --output-root {RUNS} --dry-run

In [ ]:
# 7. Run it. Re-run this cell after any disconnect.
!cd {REPO} && python scripts/run_kaggle_matrix.py --output-root {RUNS} --budget-hours 10.5

In [ ]:
# 8. Everything finished so far, read straight from Drive.
import json, glob

rows = []
for p in sorted(glob.glob(f'{RUNS}/**/results.json', recursive=True)):
    d = json.load(open(p))
    c = d['config']
    arm = c.get('arch') if c.get('arch') != 'e3dti' else c['ablation']
    rows.append((c['dataset'], c['split'], arm, c['seed'],
                 d['test']['auroc'], d['test']['auprc']))

for r in rows:
    print(f'{r[0]}/{r[1]:8s} {r[2]:16s} seed {r[3]} | TEST AUROC {r[4]:.4f} AUPRC {r[5]:.4f}')

# Mean and sample SD per arm -- the point of running three seeds.
from collections import defaultdict
import statistics

groups = defaultdict(list)
for ds, sp, arm, seed, auroc, auprc in rows:
    groups[(ds, sp, arm)].append(auroc)

print()
for key, vals in sorted(groups.items()):
    sd = statistics.stdev(vals) if len(vals) > 1 else float('nan')
    print(f'{key[0]}/{key[1]} {key[2]:16s} n={len(vals)} '
          f'AUROC {statistics.mean(vals):.4f} +/- {sd:.4f}')

## If the session dies

Reconnect and re-run cells 1-4 and 7. Everything already finished is on
Drive and is skipped; anything interrupted resumes from its last epoch.
Nothing is lost and nothing is recomputed.

Pass the same `--epochs` every time: the cosine schedule is derived from it,
so changing it mid-run changes the learning rate for the remaining epochs.